In [1]:
!pip uninstall -y langchain langchain-core langchain-community langchain-classic langchain-groq langchain-chroma langchain-huggingface chromadb

!pip install \
langchain==0.3.27 \
langchain-community==0.3.27 \
langchain-core==0.3.72 \
langchain-groq \
langchain-huggingface \
langchain-chroma \
chromadb

Found existing installation: langchain 0.3.27
Uninstalling langchain-0.3.27:
  Successfully uninstalled langchain-0.3.27
Found existing installation: langchain-core 0.3.72
Uninstalling langchain-core-0.3.72:
  Successfully uninstalled langchain-core-0.3.72
Found existing installation: langchain-community 0.3.27
Uninstalling langchain-community-0.3.27:
  Successfully uninstalled langchain-community-0.3.27
Found existing installation: langchain-groq 0.3.7
Uninstalling langchain-groq-0.3.7:
  Successfully uninstalled langchain-groq-0.3.7
Found existing installation: langchain-chroma 0.2.5
Uninstalling langchain-chroma-0.2.5:
  Successfully uninstalled langchain-chroma-0.2.5
Found existing installation: langchain-huggingface 0.3.1
Uninstalling langchain-huggingface-0.3.1:
  Successfully uninstalled langchain-huggingface-0.3.1
Found existing installation: chromadb 1.5.9
Uninstalling chromadb-1.5.9:
  Successfully uninstalled chromadb-1.5.9
  Using cached langchain-0.3.27-py3-none-any.whl.me

In [2]:
pip install pypdf

In [3]:
import chromadb
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma



loader = PyPDFLoader('python-ai.pdf')
docs = loader.load()






In [4]:
import re

def clean_text(text):
    # Join words split by hyphens at line breaks
    text = re.sub(r"-\n", "", text)

    # Replace single newlines with spaces
    text = re.sub(r"\n", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()
for doc in docs:
    doc.page_content = clean_text(doc.page_content)

print(docs[10].page_content[:1000])

Behind Python: The Languages That Power AI 11 Fig. 2.Mean peak resident set size per benchmark and language (log scale). Julia’s runtime footprint is roughly two orders of magnitude above the systems languages and is essentially constant across workloads. 4.3 Developer-Cost Metrics (M3–M5) Table 5 reports binary size, lines of code, and compilation time. Binary size spans nearly two orders of magnitude: C and C++ produce∼34KB executables, Rust 373KB (due to monomorphization and a statically linked runtime), and Go 2.5MB (including its runtime and garbage collector). Lines of code, summed across the five benchmark implementations, are surprisingly similar across languages. C++ (397) and C (406) are the most concise, while Python (406) requires essentially the same amount of code as C. Go (541) is the most verbose, largely due to explicit error handling and the use of conventional loops for numeric kernels. Compilation is sub-second per benchmark for C and C++. Rust’s8.7s cold build refl

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = splitter.split_documents(docs)


from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

collection_name = "my_rag_collection"


client = chromadb.Client()
print("Initialized in-memory ChromaDB client.")

try:


    _ = client.get_or_create_collection(name=collection_name)
    print(f"ChromaDB native client successfully got or created collection '{collection_name}' (in-memory).")
except Exception as e:
    raise RuntimeError(f"Failed to create/get ChromaDB collection with native client: {e}. This is unexpected for an in-memory client.") from e


vectorstore = Chroma(
    client=client,
    collection_name=collection_name,
    embedding_function=embeddings,
    create_collection_if_not_exists=False
)



vectorstore.add_documents(documents=texts)





Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Initialized in-memory ChromaDB client.
ChromaDB native client successfully got or created collection 'my_rag_collection' (in-memory).


['4ef3eea2-b50b-41a2-b3c6-168cb60d4da1',
 '35f0f28f-3100-4964-9f95-90d219546ba2',
 '8b660b56-1594-4b8e-813a-03ba406c0be3',
 '54460149-18fe-4fb8-8a7d-7b4223dcb73d',
 '8f9db152-0662-491b-ba31-4b295fd1b877',
 '028b7012-680f-448e-a19b-e79ee02b4d7a',
 '5ca02662-7569-4dbc-a6f0-60fe0159b3aa',
 '65641bc2-f6cc-48ee-b958-f5444f641d97',
 '5a3ffa22-2cc6-4508-ba2a-ae84848b0176',
 'f98ed9d4-4590-416e-b245-864d9a0a5cf9',
 '15b1420a-4d47-4d3c-8c4d-f174b8f48ae7',
 '010013bd-9508-40d4-ace6-d9c0bff48978',
 '4544ca8f-09c9-4a36-9871-fad68b0875aa',
 '2d2aab31-9069-4580-b014-e89319843126',
 '093744d2-dddb-400d-b8dc-43caae5644e7',
 '6d69681a-6b1f-4efe-af2e-2899dc9c267f',
 'b4a54521-94c4-4e20-aefb-508c1a64fb9d',
 'f1b5a6d1-0e18-4941-9dd9-1f409532a3a0',
 '3463ef7b-be65-4240-8e6d-83a0ae15291a',
 'ffadf5c5-f314-41b2-b5cb-6b3e4e5d7c1a',
 'c2d412aa-373e-4398-8824-6d614231fb17',
 'a27121c6-ddfb-48e5-a95b-ece329f5a2c8',
 '4982d244-6278-40c0-89ba-2501b9df7cc1',
 'fe2be83b-e8df-4631-8bc3-a2e17e4fbd68',
 '00cff6fa-bb02-

In [6]:
import os

groq_api_key = os.getenv("GROQ_API_KEY")
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile",
    temperature=0)

API Loaded Successfully!


In [7]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [8]:
import langchain
print(langchain.__version__)

0.3.27


In [9]:
from langchain.chains import RetrievalQA

In [10]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True
)

In [11]:
query = "What is Python?"

response = qa_chain.invoke({"query": query})

print(response["result"])

Python is a programming language. It is commonly used for developing artificial intelligence (AI) applications, but it relies heavily on optimized libraries written in other languages, such as C, C++, or Rust, for numerical computations.


In [12]:
import gradio as gr

def chat(question):
    response = qa_chain.invoke({"query": question})
    return response["result"]

demo = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(lines=2, placeholder="Ask anything about the PDF..."),
    outputs="text",
    title="PDF RAG Chatbot"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7d916482906958bc72.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
